<a href="https://colab.research.google.com/github/ARCHITTOMAR15/Sarcasm-Detection-with-Traditional-NLP-Deep-Learning-and-Transformers/blob/main/SARCSAM_DETECTION_USING_TRANSFORMER_MODELS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
import os
os.listdir('/content/drive/MyDrive/sarcsam_detection')

['Sarcasm_Headlines_Dataset_v2.json']

In [27]:
# import all necessary libraries
import re
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline


In [28]:
#data reading
df=pd.read_json('/content/drive/MyDrive/sarcsam_detection/Sarcasm_Headlines_Dataset_v2.json',lines=True)

In [29]:
df.head()

,is_sarcastic,headline,article_link
0,1,thirtysomething scientists unveil doomsday clo...,https://www.theonion.com/thirtysomething-scien...
1,0,dem rep. totally nails why congress is falling...,https://www.huffingtonpost.com/entry/donna-edw...
2,0,eat your veggies: 9 deliciously different recipes,https://www.huffingtonpost.com/entry/eat-your-...
3,1,inclement weather prevents liar from getting t...,https://local.theonion.com/inclement-weather-p...
4,1,mother comes pretty close to using word 'strea...,https://www.theonion.com/mother-comes-pretty-c...


In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28619 entries, 0 to 28618
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   is_sarcastic  28619 non-null  int64 
 1   headline      28619 non-null  object
 2   article_link  28619 non-null  object
dtypes: int64(1), object(2)
memory usage: 670.9+ KB


In [31]:
df.isnull().sum()

,0
is_sarcastic,0
headline,0
article_link,0


In [32]:
df.duplicated().sum()

np.int64(2)

In [33]:
df["headline"].duplicated().sum()

np.int64(116)

In [34]:
df=df.drop_duplicates(subset=["headline"]).reset_index(drop=True)

In [35]:
df["headline"].duplicated().sum()

np.int64(0)

In [36]:
# Basic minimal clearning
def clean_text(text):
  text=str(text)
  # remove URLs
  text = re.sub(r"http\S+|www\S+", "", text)
  # remove extra spaces
  text = re.sub(r"\s+", " ", text)

  return text.strip()

In [37]:
df["clean_headline"]=df["headline"].apply(clean_text)

In [38]:
df["is_sarcastic"].value_counts()

,count
is_sarcastic,
0,14951
1,13552


In [39]:
df["is_sarcastic"].value_counts(normalize=True)

,proportion
is_sarcastic,
0,0.524541
1,0.475459


In [40]:
#       Create train,test and validation slpit

In [41]:
from sklearn.model_selection import train_test_split

In [42]:
train_df,temp_df=train_test_split(df,test_size=0.20,random_state=42,stratify=df["is_sarcastic"])

In [43]:
# split temp_df into test and validation
val_df,test_df=train_test_split(temp_df,test_size=0.50,random_state=42,stratify=temp_df["is_sarcastic"])

In [44]:
#reset index
train_df=train_df.reset_index(drop=True)
test_df=test_df.reset_index(drop=True)
val_df=val_df.reset_index(drop=True)

In [45]:
print("Train_shape",train_df.shape)
print("Test_shape",test_df.shape)
print("val_shape",val_df.shape)

Train_shape (22802, 4)
Test_shape (2851, 4)
val_shape (2850, 4)


In [46]:
# convert pandas dataframe to hugging face dataset
from datasets import Dataset

In [47]:
train_dataset=Dataset.from_pandas(train_df[["clean_headline","is_sarcastic"]])
test_dataset=Dataset.from_pandas(test_df[["clean_headline","is_sarcastic"]])
val_dataset=Dataset.from_pandas(val_df[["clean_headline","is_sarcastic"]])

In [48]:
# rename is_sarcastc to label
train_dataset=train_dataset.rename_column("is_sarcastic","label")
test_dataset=test_dataset.rename_column("is_sarcastic","label")
val_dataset=val_dataset.rename_column("is_sarcastic","label")

In [49]:
## BERT MODEL PRE REQUISITE

In [50]:
#### IMPORTING TOKENIZER
from transformers import DistilBertTokenizerFast
tokenizer=DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [51]:
print(tokenizer)

DistilBertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})


In [61]:
# Tokenise the Entire Data set (Train,test,validation data set) create a function

def tokenize(batch):
  return tokenizer(batch["clean_headline"],padding='max_length',truncation=True,max_length=128)


In [63]:
train_dataset =train_dataset.map(tokenize,batched=True)
test_dataset=test_dataset.map(tokenize,batched=True)
val_dataset=val_dataset.map(tokenize,batched=True)

Map:   0%|          | 0/22802 [00:00<?, ? examples/s]

Map:   0%|          | 0/2851 [00:00<?, ? examples/s]

Map:   0%|          | 0/2850 [00:00<?, ? examples/s]

In [64]:
print(train_dataset[0])

{'clean_headline': "'gobbler games' is the brutal hunger games parody you need to see", 'label': 0, 'input_ids': [101, 1005, 2175, 11362, 2099, 2399, 1005, 2003, 1996, 12077, 9012, 2399, 12354, 2017, 2342, 2000, 2156, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [71]:
#  Convert to PYtorch Format for Batch used in Training
train_dataset.set_format('torch',columns=['input_ids','attention_mask','label'])
test_dataset.set_format('torch',columns=['input_ids','attention_mask','label'])
val_dataset.set_format('torch',columns=['input_ids','attention_mask','label'])

In [74]:
# Create Data loaders
BATCH_SIZE=16
from torch.utils.data import DataLoader
train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False)
val_loader=DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False)

In [79]:
batch=next(iter(train_loader))
print(batch.keys())

dict_keys(['label', 'input_ids', 'attention_mask'])


torch.Size([16, 128])
torch.Size([16])
torch.Size([16, 128])


In [ ]:
# LOAD PRETRAINED DISTILBERT MODEL

In [84]:
from transformers import DistilBertModel
distilbert=DistilBertModel.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
